In [ ]:
from google.cloud import bigquery
from google.oauth2 import service_account

#### Create client and link dataset Google BigQuery

In [ ]:
def create_biguery_client ():
    # Set up authentication using a service account
    credentials = service_account.Credentials.from_service_account_file('bq_key.json')
    # Create a BigQuery client using the credentials
    client = bigquery.Client(credentials=credentials, project=credentials.project_id)
    print(f"The Client:\n ${client}\n was successfully created.")
    return client


In [ ]:
client = create_biguery_client()

#### Construct a reference to a BigQuery dataset

In [ ]:
def create_dataset(client):
    # Construct a reference to the "stackoverflow" dataset
    dataset_ref = client.dataset("stackoverflow", project="bigquery-public-data")

    # API request - fetch the dataset
    dataset = client.get_dataset(dataset_ref)
    print(f"The dataset:\n ${dataset}\n has been successfully created ")
    return dataset_ref, dataset

In [ ]:
dataset, dataset_ref = create_dataset(client)

#### Explore the data

##### List the available tables

In [ ]:
# Get a list of available tables
tables = list(client.list_tables(dataset))
list_of_tables = [table.table_id for table in tables]

# Print the list of tables (uncomment print type for debug)
# print(list_of_tables)
# print(type(list_of_tables))
for table in list_of_tables:
    print(table)


#### Construct a reference to the relevant table
##### posts_answers

In [ ]:
def create_bigquery_table(client, dataset_ref):
    # Construct a reference to the "posts_answers" table
    answers_table_ref = dataset_ref.table("posts_answers")

    # API request - fetch the table
    answers_table = client.get_table(answers_table_ref)

    # Preview the first five lines of the "posts_answers" table
    print(client.list_rows(answers_table, max_results=5).to_dataframe())

In [ ]:
create_bigquery_table(client,dataset_ref)

##### posts_questions

In [ ]:
def create_bigquery_table(client, dataset_ref):
    # Construct a reference to the "posts_answers" table
    answers_table_ref = dataset_ref.table("posts_questions")

    # API request - fetch the table
    answers_table = client.get_table(answers_table_ref)

    # Preview the first five lines of the "posts_answers" table
    print(client.list_rows(answers_table, max_results=5).to_dataframe())

In [ ]:
create_bigquery_table(client,dataset_ref)

#### Construct query and validate while estimating bytes

##### post_questions table

In [ ]:
# Query that selects the id, title and owner_user_id columns from the posts_questions table. 
questions_query = """
                  SELECT id, title, owner_user_id
                  FROM `bigquery-public-data.stackoverflow.posts_questions`
                  WHERE tags Like '%bigquery%'
                """

# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(questions_query, job_config=dry_run_config)

# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

##### Join post_answers table to post_questions table

In [ ]:

# @uery that returns the id, body and owner_user_id columns from the posts_answers table for answers to "bigquery"-related questions. 
answers_query = """SELECT pa.id, pa.body, pa.owner_user_id
        FROM `bigquery-public-data.stackoverflow.posts_questions` AS pq
        INNER JOIN `bigquery-public-data.stackoverflow.posts_answers` AS pa
        ON pq.id = pa.parent_id
        WHERE pq.tags Like '%bigquery%'
"""

# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(answers_query, job_config=dry_run_config)

# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

In [ ]:
# Query that has a single row for each user who answered at least one question with a tag that includes the string "bigquery".
bigquery_experts_query = """SELECT pa.owner_user_id AS user_id, COUNT(1) AS number_of_answers
        FROM `bigquery-public-data.stackoverflow.posts_questions` AS pq
        INNER JOIN `bigquery-public-data.stackoverflow.posts_answers` AS pa
        ON pq.id = pa.parent_id
        WHERE pq.tags Like '%bigquery%'
        GROUP BY user_id"""

# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(bigquery_experts_query, job_config=dry_run_config)

# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

#### Build Query into general function used for website

In [ ]:
def expert_finder(topic, client):
    my_query = f"""
               SELECT a.owner_user_id AS user_id, COUNT(1) AS number_of_answers
               FROM `bigquery-public-data.stackoverflow.posts_questions` AS q
               INNER JOIN `bigquery-public-data.stackoverflow.posts_answers` AS a
                   ON q.id = a.parent_Id
               WHERE q.tags like '%{topic}%'
               GROUP BY a.owner_user_id
    """
    # Create a "QueryJobConfig" object to set the "dry_run"
    dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

    # API request - dry run query to estimate cost of query
    dry_run_query_job = client.query(my_query, job_config=dry_run_config)
    print(my_query)

    # Print estimated number of bytes processed by the query
    print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

        # Set up the query (a real service would have good error handling for 
    # queries that scan too much data)
    safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)      
    my_query_job = client.query(my_query, job_config=safe_config)
    
    # API request - run the query, and return a pandas DataFrame
    results = my_query_job.to_dataframe()
    # print(results)
    return results

#### Choose topic to research from Stack Overflow

In [ ]:
topic = "pytorch"
expert_finder(topic, client)